# 📓 Notebook 4: Error Analysis – Phân Tích Lỗi Chuyên Sâu
**Nhóm Mù Công Nghệ** | Đề tài 22 | Tuần 4  
Mục tiêu: Phân tích FP/FN từng nhãn, xác định nguyên nhân, đề xuất cải thiện

## 1. Import & setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pickle, json, re, os, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (
    classification_report, f1_score, hamming_loss,
    confusion_matrix, roc_auc_score, average_precision_score
)

LABELS       = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("✅ Import xong")


## 2. Tải dữ liệu & huấn luyện nhanh LR+Threshold
> Nếu đã có file pkl từ Notebook 1 & 3, có thể load trực tiếp bên dưới

In [ ]:
def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'[^a-z0-9\s!?.,\'\-]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

df_raw = pd.read_csv('train.csv')
df_raw['clean_text'] = df_raw['comment_text'].apply(preprocess)

X = df_raw['clean_text'].values
y = df_raw[LABELS].values

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.125, random_state=RANDOM_STATE)

# Lưu raw text tương ứng với test để phân tích
_, test_raw_idx = train_test_split(df_raw.index, test_size=0.2, random_state=RANDOM_STATE)
df_test = df_raw.loc[test_raw_idx].reset_index(drop=True)

tfidf = TfidfVectorizer(max_features=50_000, ngram_range=(1,2), sublinear_tf=True, min_df=3)
X_train_tf = tfidf.fit_transform(X_train)
X_test_tf  = tfidf.transform(X_test)

lr = OneVsRestClassifier(
    LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=RANDOM_STATE), n_jobs=-1
)
lr.fit(X_train_tf, y_train)
y_proba = lr.predict_proba(X_test_tf)
print("✅ Đã huấn luyện và lấy xác suất")


In [ ]:
# Load threshold từ Notebook 3 (hoặc dùng giá trị mặc định)
try:
    with open('results/best_thresholds.json') as f:
        best_thresholds = json.load(f)
    print("✅ Loaded thresholds:", best_thresholds)
except FileNotFoundError:
    best_thresholds = {
        'toxic': 0.35, 'severe_toxic': 0.30, 'obscene': 0.38,
        'threat': 0.40, 'insult': 0.36, 'identity_hate': 0.30
    }
    print("⚠️  Dùng threshold mặc định:", best_thresholds)

thresh_arr   = np.array([best_thresholds[l] for l in LABELS])
y_pred       = (y_proba >= thresh_arr).astype(int)
y_pred_base  = (y_proba >= 0.5).astype(int)


## 3. Tổng quan lỗi theo nhãn

In [ ]:
error_summary = []
for i, lbl in enumerate(LABELS):
    TP = ((y_test[:, i] == 1) & (y_pred[:, i] == 1)).sum()
    TN = ((y_test[:, i] == 0) & (y_pred[:, i] == 0)).sum()
    FP = ((y_test[:, i] == 0) & (y_pred[:, i] == 1)).sum()
    FN = ((y_test[:, i] == 1) & (y_pred[:, i] == 0)).sum()
    precision = TP / (TP + FP + 1e-9)
    recall    = TP / (TP + FN + 1e-9)
    f1        = 2 * precision * recall / (precision + recall + 1e-9)
    error_summary.append({
        'Nhãn': lbl, 'TP': TP, 'TN': TN, 'FP': FP, 'FN': FN,
        'Precision': round(precision, 3), 'Recall': round(recall, 3), 'F1': round(f1, 3),
        'Support (+)': int(y_test[:, i].sum())
    })

df_err = pd.DataFrame(error_summary)
print(df_err.to_string(index=False))


## 4. Biểu đồ FP/FN theo nhãn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# FP & FN
x = np.arange(len(LABELS)); w = 0.35
ax = axes[0]
fp_vals = df_err['FP'].values
fn_vals = df_err['FN'].values
ax.bar(x - w/2, fp_vals, w, label='False Positive (FP)', color='tomato',    alpha=0.85)
ax.bar(x + w/2, fn_vals, w, label='False Negative (FN)', color='steelblue', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=25)
ax.set_title('FP và FN theo nhãn – LR + Threshold Tuning', fontweight='bold')
ax.set_ylabel('Số lượng'); ax.legend(); ax.grid(axis='y', alpha=0.3)

# Precision / Recall / F1
ax2 = axes[1]
width = 0.25
xs = np.arange(len(LABELS))
ax2.bar(xs - width, df_err['Precision'], width, label='Precision', color='seagreen', alpha=0.85)
ax2.bar(xs,         df_err['Recall'],    width, label='Recall',    color='steelblue', alpha=0.85)
ax2.bar(xs + width, df_err['F1'],        width, label='F1',        color='darkorange', alpha=0.85)
ax2.set_xticks(xs); ax2.set_xticklabels(LABELS, rotation=25)
ax2.set_title('Precision / Recall / F1 từng nhãn', fontweight='bold')
ax2.set_ylabel('Score'); ax2.set_ylim(0, 1); ax2.legend(); ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('error_analysis_overview.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Hàm trích xuất mẫu lỗi thật

In [ ]:
def get_error_samples(label, error_type='FN', n=10, proba_sort=True):
    """
    Trích xuất mẫu lỗi thật từ tập test.
    label      : tên nhãn (trong LABELS)
    error_type : 'FN' (bỏ sót) hoặc 'FP' (cảnh báo nhầm)
    n          : số mẫu muốn lấy
    """
    i = LABELS.index(label)
    if error_type == 'FN':
        # Dự đoán 0 nhưng thật ra là 1 → bỏ sót
        mask = (y_test[:, i] == 1) & (y_pred[:, i] == 0)
    else:  # FP
        # Dự đoán 1 nhưng thật ra là 0 → cảnh báo nhầm
        mask = (y_test[:, i] == 0) & (y_pred[:, i] == 1)

    idx = np.where(mask)[0]
    if len(idx) == 0:
        print(f"Không có mẫu {error_type} cho nhãn {label}")
        return pd.DataFrame()

    probas = y_proba[idx, i]
    if proba_sort:
        # FN: xác suất cao nhất (gần được nhận) → dễ cải thiện nhất
        # FP: xác suất cao nhất (mô hình "tin" nhất) → nguy hiểm nhất
        sort_idx = np.argsort(probas)[::-1]
        idx = idx[sort_idx]

    result = []
    for j in idx[:n]:
        result.append({
            'comment'       : df_test.iloc[j]['comment_text'][:250],
            f'true_{label}' : int(y_test[j, i]),
            f'pred_{label}' : int(y_pred[j, i]),
            f'proba_{label}': round(float(y_proba[j, i]), 4),
            'all_true_labels': [LABELS[k] for k in range(6) if y_test[j, k] == 1],
        })
    return pd.DataFrame(result)

# Test thử
print("== Mẫu FN của nhãn 'threat' ==")
df_fn_threat = get_error_samples('threat', 'FN', n=5)
for _, row in df_fn_threat.iterrows():
    print(f"  [proba={row['proba_threat']:.3f}] {row['comment'][:150]}")
    print(f"  True labels: {row['all_true_labels']}\n")


## 6. Phân tích FN từng nhãn hiếm

In [ ]:
for lbl in ['threat', 'identity_hate', 'severe_toxic']:
    print(f"\n{'='*70}")
    print(f"  FALSE NEGATIVES – Nhãn: {lbl.upper()}")
    print(f"{'='*70}")
    df_fn = get_error_samples(lbl, 'FN', n=8)
    if df_fn.empty: continue
    for _, row in df_fn.iterrows():
        prob_key = f'proba_{lbl}'
        print(f"  Prob={row[prob_key]:.3f} | {row['comment'][:200]}")
        print(f"  True labels: {row['all_true_labels']}")
        print()


## 7. Phân tích FP từng nhãn

In [ ]:
for lbl in ['toxic', 'obscene', 'insult']:
    print(f"\n{'='*70}")
    print(f"  FALSE POSITIVES – Nhãn: {lbl.upper()}")
    print(f"{'='*70}")
    df_fp = get_error_samples(lbl, 'FP', n=5)
    if df_fp.empty: continue
    for _, row in df_fp.iterrows():
        prob_key = f'proba_{lbl}'
        print(f"  Prob={row[prob_key]:.3f} | {row['comment'][:200]}")
        print(f"  True labels: {row['all_true_labels']}")
        print()


## 8. Phân tích từ khóa gây lỗi (TF-IDF Feature Importance)

In [ ]:
def top_features(label, n=20):
    """Từ có trọng số TF-IDF cao nhất cho nhãn lbl."""
    i   = LABELS.index(label)
    clf = lr.estimators_[i]  # LogisticRegression của nhãn i
    coef = clf.coef_[0]
    vocab = tfidf.get_feature_names_out()
    top_pos = np.argsort(coef)[::-1][:n]   # từ → toxic
    top_neg = np.argsort(coef)[:n]         # từ → non-toxic

    print(f"\n[{label}] TOP {n} TỪ TĂNG XÁC SUẤT NHÃN:")
    print("  " + " | ".join([f"{vocab[j]} ({coef[j]:.2f})" for j in top_pos]))
    print(f"\n[{label}] TOP {n} TỪ GIẢM XÁC SUẤT NHÃN:")
    print("  " + " | ".join([f"{vocab[j]} ({coef[j]:.2f})" for j in top_neg]))
    return vocab, coef, top_pos, top_neg

for lbl in LABELS:
    top_features(lbl, n=15)


In [ ]:
# Vẽ feature importance cho 2 nhãn khó nhất
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, lbl in zip(axes, ['threat', 'identity_hate']):
    i   = LABELS.index(lbl)
    clf = lr.estimators_[i]
    coef = clf.coef_[0]
    vocab = tfidf.get_feature_names_out()
    top20 = np.argsort(np.abs(coef))[::-1][:20]
    words  = [vocab[j] for j in top20]
    values = [coef[j]  for j in top20]
    colors = ['tomato' if v > 0 else 'steelblue' for v in values]
    ax.barh(words[::-1], values[::-1], color=colors[::-1], alpha=0.85, edgecolor='white')
    ax.axvline(0, color='black', lw=0.8)
    ax.set_title(f'Feature Importance – {lbl}', fontweight='bold')
    ax.set_xlabel('LR Coefficient')
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('feature_importance_threat_identityhate.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Phân bố xác suất FN vs TP (để hiểu ngưỡng)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for i, (lbl, ax) in enumerate(zip(LABELS, axes.flatten())):
    t = best_thresholds[lbl]
    # TP: đúng nhãn, dự đoán đúng
    mask_tp = (y_test[:, i] == 1) & (y_pred[:, i] == 1)
    # FN: đúng nhãn, dự đoán sai
    mask_fn = (y_test[:, i] == 1) & (y_pred[:, i] == 0)

    if mask_tp.sum() > 0:
        ax.hist(y_proba[mask_tp, i], bins=30, alpha=0.6, color='seagreen', label=f'TP (n={mask_tp.sum()})', density=True)
    if mask_fn.sum() > 0:
        ax.hist(y_proba[mask_fn, i], bins=30, alpha=0.6, color='tomato',   label=f'FN (n={mask_fn.sum()})', density=True)

    ax.axvline(t,   color='crimson', linestyle='--', lw=2, label=f'threshold={t}')
    ax.axvline(0.5, color='gray',    linestyle=':',  lw=1.5, label='t=0.5')
    ax.set_title(f'{lbl}', fontweight='bold')
    ax.set_xlabel('Predicted Probability'); ax.set_ylabel('Density')
    ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.suptitle('Phân bố xác suất dự đoán: TP vs FN theo từng nhãn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('proba_distribution_tp_fn.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Xuất bảng lỗi chi tiết ra CSV

In [ ]:
os.makedirs('results', exist_ok=True)
all_errors = []

for lbl in LABELS:
    for etype in ['FN', 'FP']:
        df_e = get_error_samples(lbl, etype, n=20)
        if not df_e.empty:
            df_e.insert(0, 'label', lbl)
            df_e.insert(1, 'error_type', etype)
            all_errors.append(df_e)

df_all_errors = pd.concat(all_errors, ignore_index=True)
df_all_errors.to_csv('results/error_analysis.csv', index=False, encoding='utf-8-sig')
print(f"✅ Đã lưu {len(df_all_errors)} mẫu lỗi → results/error_analysis.csv")
df_all_errors.head(10)


## 11. Tổng kết phân tích lỗi

In [ ]:
print("=" * 70)
print("TỔNG KẾT PHÂN TÍCH LỖI – Nhóm Mù Công Nghệ")
print("=" * 70)

print("\n1. MÔ HÌNH SAI Ở ĐÂU?")
for _, row in df_err.sort_values('FN', ascending=False).iterrows():
    print(f"   {row['Nhãn']:<16}: FN={row['FN']:4d}, FP={row['FP']:4d} | F1={row['F1']:.3f}")

print("\n2. NGUYÊN NHÂN CHÍNH:")
reasons = {
    'threat'       : "Đe dọa gián tiếp, không có từ khóa rõ; TF-IDF không nắm ngữ cảnh",
    'identity_hate': "Thù ghét ẩn, dùng hàm ý phân biệt, không có từ tục tĩu trực tiếp",
    'severe_toxic' : "Nội dung cực đoan không tục tĩu rõ; nhãn chồng chéo với toxic",
    'obscene'      : "Biến thể viết tắt từ ngữ tục, leet speak; TF-IDF bỏ sót",
    'insult'       : "Sarcasm và ngôn ngữ châm biếm gián tiếp",
    'toxic'        : "FP cao do từ ngữ mạnh trong bình luận nghiêm túc (ví dụ phê bình)",
}
for lbl, reason in reasons.items():
    print(f"   {lbl:<16}: {reason}")

print("\n3. CÁC GIẢI PHÁP ĐÃ ÁP DỤNG:")
print("   ✅ Per-label threshold tuning → Macro-F1: 0.49 → 0.59 (+20%)")
print("   ✅ Fine-tune DistilBERT → Macro-F1: 0.49 → 0.66 (+35%)")
print("   ✅ Class weights trong BCELoss → cải thiện nhãn hiếm")

print("\n4. ĐỀ XUẤT TUẦN 5:")
print("   → Char n-gram TF-IDF (bắt biến thể từ ngữ toxic)")
print("   → Back-translation augmentation cho threat, identity_hate")
print("   → Attention visualization để giải thích dự đoán")
print("   → Ensemble LR + DistilBERT")
